In [7]:
# ========================================================================
# SETUP DE PATHS - USAR source/config.py
# ========================================================================

import sys
from pathlib import Path

notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    sys.path.insert(0, str(notebook_dir.parent))
else:
    sys.path.insert(0, str(notebook_dir))

from source.config import (
    PROJ_ROOT, RAW_DATA_DIR, PROCESSED_DATA_DIR, EXTERNAL_DATA_DIR,
    get_data_path
)

from loguru import logger

logger.info(f"📁 Projeto: {PROJ_ROOT}")

# ========================================================================

2026-03-28 15:25:50.940 | INFO     | __main__:<module>:21 - 📁 Projeto: C:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering


# indice

- [1 Load libs](#1-Load-libs)
- [2 Config path](#2-Config-path)


## 1 Load libs

In [8]:
import pandas as pd

# Configuração para exibir todas as colunas
pd.set_option('display.max_columns', None)

# Configuração para exibir todas as linhas
pd.set_option('display.max_rows', None)

# Configuração para que o conteúdo de uma coluna não seja cortado
pd.set_option('display.max_colwidth', None)

# Configuração para expandir a largura da exibição para que mais colunas caibam na tela
pd.set_option('display.width', 1000)


## 2 Config path

In [9]:
# Caminhos dinâmicos usando config.py (não hardcoded)
path_accounts_df = get_data_path("HI-Small_accounts.csv", "external")
path_trans_df = get_data_path("HI-Small_Trans.csv", "external")

logger.info(f"Accounts: {path_accounts_df}")
logger.info(f"Trans: {path_trans_df}")

2026-03-28 15:25:50.986 | INFO     | __main__:<module>:5 - Accounts: C:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering\data\external\HI-Small_accounts.csv
2026-03-28 15:25:50.987 | INFO     | __main__:<module>:6 - Trans: C:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering\data\external\HI-Small_Trans.csv


## 3 Union datasets

In [10]:


# Carregar os arquivos
accounts_df = pd.read_csv(path_accounts_df)
trans_df = pd.read_csv(path_trans_df)

# Renomear colunas duplicadas em trans_df para maior clareza
trans_df.columns = ['Timestamp', 'From Bank', 'From Account', 'To Bank', 'To Account', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']

# 1. Juntar transações com informações da conta de origem (remetente)
trans_enriched_df = pd.merge(
    trans_df,
    accounts_df,
    left_on=['From Bank', 'From Account'],
    right_on=['Bank ID', 'Account Number'],
    how='left'
)

# Renomear colunas para evitar conflitos
trans_enriched_df = trans_enriched_df.rename(columns={
    'Bank Name': 'From Bank Name',
    'Entity ID': 'From Entity ID',
    'Entity Name': 'From Entity Name'
})

# 2. Juntar o resultado com informações da conta de destino (destinatário)
trans_enriched_df = pd.merge(
    trans_enriched_df,
    accounts_df,
    left_on=['To Bank', 'To Account'],
    right_on=['Bank ID', 'Account Number'],
    how='left',
    suffixes=('', '_To')
)

# Renomear colunas para maior clareza
trans_enriched_df = trans_enriched_df.rename(columns={
    'Bank Name': 'To Bank Name',
    'Entity ID': 'To Entity ID',
    'Entity Name': 'To Entity Name'
})

# Exibir o resultado
print("Tabela de Transações Enriquecida:")
print(trans_enriched_df.head())


Tabela de Transações Enriquecida:
          Timestamp  From Bank From Account  To Bank To Account  Amount Received Receiving Currency  Amount Paid Payment Currency Payment Format  Is Laundering               From Bank Name  Bank ID Account Number From Entity ID        From Entity Name                 To Bank Name  Bank ID_To Account Number_To To Entity ID          To Entity Name
0  2022/09/01 00:20         10    8000EBD30       10  8000EBD30          3697.34          US Dollar      3697.34        US Dollar   Reinvestment              0     National Bank of Laramie       10      8000EBD30      800D232D0          Partnership #1     National Bank of Laramie          10         8000EBD30    800D232D0          Partnership #1
1  2022/09/01 00:20       3208    8000F4580        1  8000F5340             0.01          US Dollar         0.01        US Dollar         Cheque              0       Sappo Cooperative Bank     3208      8000F4580      8008EEA70          Partnership #2           Arbor Sa

## 3 Salve data raw

In [11]:
# Salvar usando config.py (não hardcoded path)
output_path = get_data_path('trans_enriched.csv', 'raw')
trans_enriched_df.to_csv(output_path, index=False)

logger.success(f"✅ Dados salvos em: {output_path}")

2026-03-28 15:26:45.602 | SUCCESS  | __main__:<module>:5 - ✅ Dados salvos em: C:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering\data\raw\trans_enriched.csv


In [12]:
trans_enriched_df.head()

,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,From Bank Name,Bank ID,Account Number,From Entity ID,From Entity Name,To Bank Name,Bank ID_To,Account Number_To,To Entity ID,To Entity Name
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0,National Bank of Laramie,10,8000EBD30,800D232D0,Partnership #1,National Bank of Laramie,10,8000EBD30,800D232D0,Partnership #1
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0,Sappo Cooperative Bank,3208,8000F4580,8008EEA70,Partnership #2,Arbor Savings Bank,1,8000F5340,800AA5D20,Corporation #1
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0,National Bank of Fort Wayne,3209,8000F4670,800FBB3A0,Partnership #3,National Bank of Fort Wayne,3209,8000F4670,800FBB3A0,Partnership #3
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0,National Bank of the East,12,8000F5030,800C0EF20,Sole Proprietorship #1,National Bank of the East,12,8000F5030,800C0EF20,Sole Proprietorship #1
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0,National Bank of Laramie,10,8000F5200,800C3EC10,Partnership #4,National Bank of Laramie,10,8000F5200,800C3EC10,Partnership #4
